# V2D Challenge CPU Reconstruction Comparison

This notebook runs bounded classical baselines for four requested paradigms. Heavy GPU models are checked and logged; deterministic CPU proxies keep every block executable and make the failure boundary visible. Track 1 has no released per-frame ground truth in this local revision, while Track 3 has mocap poses filtered by `visible`.

### Setup and sequence selection

**What it does.** Selects one short Track 1 video and one short Track 3 ego video from actual metadata.

**Input.** Local V2D files.

**Output.** `outputs/reconstruction_selection.json`.

**References.** Track 1/3 README files.

In [3]:
from pathlib import Path
import json, sys, time, platform, subprocess, importlib.util
import matplotlib; matplotlib.use('Agg', force=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from IPython.display import display, Image
ROOT=Path.cwd()
if not (ROOT/'v2d_cpu_utils.py').exists(): ROOT=Path('/Users/luran/Desktop/V2D_Challenge')
sys.path.insert(0,str(ROOT))
from v2d_cpu_utils import *
paths=ensure_outputs()
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'font.size':9})

t1=track1_paths(); t3=track3_paths(); a=min(t1,key=lambda x:x['frame_count']); b=min(t3,key=lambda x:len(read_parquet(x['parquet']))); f1,i1=sample_video(a['video'],max_frames=24,resize_width=640); f3,i3=sample_video(b['videos']['ego_cam_a'],max_frames=24,resize_width=640); sel={'track1':{'episode_index':a['episode_index'],'sequence_id':a.get('sequence_id'),'object':a.get('object'),'video':a['video'],'frames':len(f1),'fps':i1['fps']},'track3':{'episode_index':b['episode_index'],'task':b.get('task'),'video':b['videos']['ego_cam_a'],'frames':len(f3),'fps':i3['fps']},'matching_note':'Track 1 and Track 3 releases have disjoint episode IDs and object sets; this is a representative per-track comparison, not a paired benchmark.'}; write_json(paths['root']/'reconstruction_selection.json',sel); print(json.dumps(sel,indent=2))

{
  "track1": {
    "episode_index": 16,
    "sequence_id": "2026-09-11_15-12-36_foam_grass_block_rotate_04",
    "object": "foam_grass_block",
    "video": "/Users/luran/Desktop/V2D_Challenge/video_to_data_challenge/track_1/videos/chunk-000/observation.images.exo_camera/episode_000016.mp4",
    "frames": 24,
    "fps": 30.0
  },
  "track3": {
    "episode_index": 11,
    "task": "Cup_Stack",
    "video": "/Users/luran/Desktop/V2D_Challenge/video_to_data_challenge/track_3/public/videos/chunk-000/observation.images.ego_cam_a/episode_000011.mp4",
    "frames": 24,
    "fps": 20.0
  },
  "matching_note": "Track 1 and Track 3 releases have disjoint episode IDs and object sets; this is a representative per-track comparison, not a paired benchmark."
}


### M1 - feature-based SfM attempt

**What it does.** Detects ORB features and matches adjacent real frames, and checks for `pycolmap`. No unbounded dense reconstruction is attempted.

**Input.** Sampled Track 1 and Track 3 frames.

**Output.** `outputs/logs/m1_sfm_attempt.json`, `outputs/figs/m1_feature_tracks.png`.

**References.** COLMAP: https://colmap.github.io/ ; Schönberger and Frahm, CVPR 2016.

In [5]:
from pathlib import Path
import json, sys, time, platform, subprocess, importlib.util
import matplotlib; matplotlib.use('Agg', force=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from IPython.display import display, Image
ROOT=Path.cwd()
if not (ROOT/'v2d_cpu_utils.py').exists(): ROOT=Path('/Users/luran/Desktop/V2D_Challenge')
sys.path.insert(0,str(ROOT))
from v2d_cpu_utils import *
paths=ensure_outputs()
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'font.size':9})

def orb(frames):
 o=cv2.ORB_create(nfeatures=800); bf=cv2.BFMatcher(cv2.NORM_HAMMING,crossCheck=True); counts=[]; tracks=[]; prev=None; kp0=None
 for fr in frames:
  kp,des=o.detectAndCompute(cv2.cvtColor(fr,cv2.COLOR_BGR2GRAY),None)
  if prev is not None and des is not None and prev is not None: ms=sorted(bf.match(prev,des),key=lambda m:m.distance); counts.append(len(ms)); tracks.append([(kp0[m.queryIdx].pt,kp[m.trainIdx].pt) for m in ms[:100]])
  kp0,prev=kp,des
 return {'adjacent_pairs':max(0,len(frames)-1),'median_matches':float(np.median(counts)) if counts else 0.0,'min_matches':int(min(counts)) if counts else 0,'max_matches':int(max(counts)) if counts else 0},tracks
t0=time.perf_counter(); s1,tr1=orb(f1); s3,tr3=orb(f3); elapsed=time.perf_counter()-t0
try:
 import pycolmap; pc='importable; full reconstruction bounded out because camera intrinsics/masks/extrinsics are incomplete.'
except Exception as exc: pc=f'unavailable: {type(exc).__name__}: {exc}'
m1={'status':'feature tracks only; metric SfM not claimed','track1':s1,'track3':s3,'pycolmap':pc,'wall_time_s':elapsed,'peak_rss_mb':peak_rss_mb()}; write_json(paths['logs']/'m1_sfm_attempt.json',m1); fig,ax=plt.subplots(1,2,figsize=(10,4));
for q,fr,tr,title in [(ax[0],f1[0],tr1[0] if tr1 else [],'Track 1 ORB'),(ax[1],f3[0],tr3[0] if tr3 else [],'Track 3 ORB')]:
 q.imshow(cv2.cvtColor(fr,cv2.COLOR_BGR2RGB)); [q.plot([p[0],r[0]],[p[1],r[1]],'-',alpha=.35,lw=.6) for p,r in tr[:40]]; q.set_title(title); q.axis('off')
fig.tight_layout(); fig.savefig(paths['figs']/'m1_feature_tracks.png',dpi=140); plt.close(fig); display(pd.DataFrame([{'method':'M1 SfM','track':'Track 1',**s1,'status':'bounded feature-only'},{'method':'M1 SfM','track':'Track 3',**s3,'status':'bounded feature-only'}]))

,method,track,adjacent_pairs,median_matches,min_matches,max_matches,status
0,M1 SfM,Track 1,23,711.0,570,742,bounded feature-only
1,M1 SfM,Track 3,23,572.0,461,632,bounded feature-only


### M2 - calibrated silhouette back-projection proxy

**What it does.** Uses CPU background subtraction and connected components for a reproducible moving foreground mask, estimates pseudo-depth from component size, and back-projects mask pixels into a sparse point cloud. This is the CPU fallback for Depth Anything V2 or MoGe when weights are unavailable.

**Input.** Sampled frames.

**Output.** `outputs/meshes/m2_track1_points.npy`, `m2_track3_points.npy`, coverage figure, and timing log.

**References.** Depth Anything V2: https://github.com/DepthAnything/Depth-Anything-V2 ; MoGe: https://github.com/microsoft/MoGe.

In [7]:
from pathlib import Path
import json, sys, time, platform, subprocess, importlib.util
import matplotlib; matplotlib.use('Agg', force=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from IPython.display import display, Image
ROOT=Path.cwd()
if not (ROOT/'v2d_cpu_utils.py').exists(): ROOT=Path('/Users/luran/Desktop/V2D_Challenge')
sys.path.insert(0,str(ROOT))
from v2d_cpu_utils import *
paths=ensure_outputs()
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'font.size':9})

def masks(frames):
 bg=cv2.createBackgroundSubtractorMOG2(history=max(20,len(frames)),varThreshold=24,detectShadows=False); out=[]
 for fr in frames:
  m=(bg.apply(fr)>32).astype(np.uint8)*255; m=cv2.medianBlur(m,5); m=cv2.morphologyEx(m,cv2.MORPH_OPEN,np.ones((3,3),np.uint8)); out.append(m)
 return out
def back(frames,ms):
 pts=[]; cov=[]
 for fr,m in zip(frames,ms):
  y,x=np.where(m>0); cov.append(float(len(x)/m.size))
  if len(x)<20: continue
  x0,x1,y0,y1=x.min(),x.max(),y.min(),y.max(); z=.45*fr.shape[0]/max(2,y1-y0+1); fx=.9*fr.shape[1]; keep=np.linspace(0,len(x)-1,min(600,len(x))).astype(int); xx=x[keep]; yy=y[keep]; pts.append(np.c_[(xx-fr.shape[1]/2)*z/fx,(yy-fr.shape[0]/2)*z/fx,np.full(len(xx),z)])
 return np.vstack(pts) if pts else np.empty((0,3)),cov
t0=time.perf_counter(); msk1=masks(f1); p1,c1=back(f1,msk1); msk3=masks(f3); p3,c3=back(f3,msk3); elapsed=time.perf_counter()-t0; np.save(paths['meshes']/'m2_track1_points.npy',p1); np.save(paths['meshes']/'m2_track3_points.npy',p3); fig,ax=plt.subplots(1,2,figsize=(9,3)); ax[0].plot(c1); ax[1].plot(c3,color='tab:orange'); [q.set_xlabel('sample index') for q in ax]; [q.set_ylabel('foreground fraction') for q in ax]; fig.tight_layout(); fig.savefig(paths['figs']/'m2_mask_coverage.png',dpi=140); plt.close(fig); m2={'status':'CPU mask plus pseudo-depth proxy; Depth Anything V2/MoGe weights not local','wall_time_s':elapsed,'peak_rss_mb':peak_rss_mb(),'track1':{'points':len(p1),'mean_mask_fraction':float(np.mean(c1))},'track3':{'points':len(p3),'mean_mask_fraction':float(np.mean(c3))}}; write_json(paths['logs']/'m2_depth_proxy.json',m2); display(pd.DataFrame([{'track':'Track 1',**m2['track1']},{'track':'Track 3',**m2['track3']}]))

,track,points,mean_mask_fraction
0,Track 1,8975,0.045686
1,Track 3,14400,0.170461


### M3 - feed-forward single-view mesh availability and proxy

**What it does.** Checks TripoSR/InstantMesh availability and exports a deterministic ellipsoid proxy from observed point-cloud extents when weights are not bundled.

**Input.** M2 point clouds.

**Output.** `outputs/meshes/m3_track1_proxy.ply`, `m3_track3_proxy.ply`, and an explicit failure log.

**References.** TripoSR: https://github.com/VAST-AI-Research/TripoSR ; InstantMesh: https://github.com/TencentARC/InstantMesh.

In [9]:
from pathlib import Path
import json, sys, time, platform, subprocess, importlib.util
import matplotlib; matplotlib.use('Agg', force=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from IPython.display import display, Image
ROOT=Path.cwd()
if not (ROOT/'v2d_cpu_utils.py').exists(): ROOT=Path('/Users/luran/Desktop/V2D_Challenge')
sys.path.insert(0,str(ROOT))
from v2d_cpu_utils import *
paths=ensure_outputs()
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'font.size':9})

def write_ellipsoid_ply(points,path):
    if len(points): center=points.mean(0); ext=np.maximum(points.max(0)-points.min(0),1e-3)/2
    else: center=np.zeros(3); ext=np.ones(3)*.05
    # ASCII PLY ellipsoid proxy: no optional mesh package is required.
    u=np.linspace(0,np.pi,12); v=np.linspace(0,2*np.pi,24,endpoint=False)
    verts=[]
    for a in u:
        for b0 in v: verts.append(center+ext*np.array([np.sin(a)*np.cos(b0),np.sin(a)*np.sin(b0),np.cos(a)]))
    faces=[]; nu,nv=len(u),len(v)
    for i in range(nu-1):
        for j in range(nv):
            j2=(j+1)%nv; a=i*nv+j; b=i*nv+j2; c0=(i+1)*nv+j2; d=(i+1)*nv+j; faces.extend([(a,b,c0),(a,c0,d)])
    with open(path,'w',encoding='ascii') as f:
        f.write(f'ply\nformat ascii 1.0\nelement vertex {len(verts)}\nproperty float x\nproperty float y\nproperty float z\nelement face {len(faces)}\nproperty list uchar int vertex_indices\nend_header\n')
        for q in verts: f.write(f'{q[0]} {q[1]} {q[2]}\n')
        for q in faces: f.write(f'3 {q[0]} {q[1]} {q[2]}\n')
    return {'vertices':len(verts),'faces':len(faces),'extents_m':ext.tolist()}

avail={}
for mod in ['tsr','instant_mesh','huggingface_hub']:
 try: __import__(mod); avail[mod]='importable'
 except Exception as exc: avail[mod]=f'unavailable: {type(exc).__name__}: {exc}'
t0=time.perf_counter(); q1=write_ellipsoid_ply(p1,paths['meshes']/'m3_track1_proxy.ply'); q3=write_ellipsoid_ply(p3,paths['meshes']/'m3_track3_proxy.ply'); elapsed=time.perf_counter()-t0; m3={'model_availability':avail,'status':'ellipsoid proxy exported because TripoSR/InstantMesh weights are not local and network is disabled','wall_time_s':elapsed,'peak_rss_mb':peak_rss_mb(),'track1':q1,'track3':q3}; write_json(paths['logs']/'m3_single_view_attempt.json',m3); display(pd.DataFrame([{'method':'M3 TripoSR/InstantMesh','track':'Track 1','status':m3['status'],'vertices':q1['vertices']},{'method':'M3 TripoSR/InstantMesh','track':'Track 3','status':m3['status'],'vertices':q3['vertices']}]))

,method,track,status,vertices
0,M3 TripoSR/InstantMesh,Track 1,ellipsoid proxy exported because TripoSR/Insta...,288
1,M3 TripoSR/InstantMesh,Track 3,ellipsoid proxy exported because TripoSR/Insta...,288


### M4 - known-CAD pose baseline on Track 3

**What it does.** Loads the released Track 3 mesh/URDF identity and evaluates a CPU-safe CAD initialization baseline: hold the first visible pose constant. The release has no camera-to-mocap-world extrinsic, so image-to-world ICP cannot be initialized honestly.

**Input.** Track 3 mocap poses, visibility flags, mesh/URDF files.

**Output.** `outputs/metrics/m4_track3_pose_errors.csv`, error curve, and CAD log.

**References.** Track 3 pose convention; Besl and McKay, ICP, IEEE TPAMI 1992.

In [11]:
from pathlib import Path
import json, sys, time, platform, subprocess, importlib.util
import matplotlib; matplotlib.use('Agg', force=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from IPython.display import display, Image
ROOT=Path.cwd()
if not (ROOT/'v2d_cpu_utils.py').exists(): ROOT=Path('/Users/luran/Desktop/V2D_Challenge')
sys.path.insert(0,str(ROOT))
from v2d_cpu_utils import *
paths=ensure_outputs()
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'font.size':9})

d=read_parquet(b['parquet']); objs=decode_track3_objects(d); rows=[]; curves=[]
for name,v in objs.items():
 valid=v['visible']; pose=v['pose']; ix=np.where(valid)[0]
 if not len(ix): continue
 first=ix[0]; pred=np.repeat(pose[first:first+1],len(pose),axis=0); te=np.linalg.norm(pred[:,:3]-pose[:,:3],axis=1); ae=np.full(len(pose),np.nan); ae[valid]=np.degrees(quat_angle_rad(pred[valid,3:],pose[valid,3:]))
 for i in ix: rows.append({'object':name,'frame':int(i),'translation_error_m':float(te[i]),'orientation_error_deg':float(ae[i]),'visible':True})
 curves.append((name,te,valid))
m4df=pd.DataFrame(rows); m4df.to_csv(paths['metrics']/'m4_track3_pose_errors.csv',index=False); fig,ax=plt.subplots(2,1,figsize=(8,5),sharex=True); [ax[0].plot(te,label=n) for n,te,v in curves]; [ax[1].plot(np.where(v,1,np.nan),label=n) for n,te,v in curves]; ax[0].set_ylabel('translation error (m)'); ax[1].set_ylabel('visible'); ax[1].set_xlabel('frame index'); ax[0].legend(ncol=3,fontsize=8); fig.tight_layout(); fig.savefig(paths['figs']/'m4_track3_errors.png',dpi=140); plt.close(fig); 
# Visibility-bucket audit across all public Track 3 episodes. Invisible rows are excluded from pose errors.
bucket_rows=[]
for rr in track3_paths():
    dd=read_parquet(rr['parquet']); oo=decode_track3_objects(dd)
    for nn,vv in oo.items():
        valid=vv['visible']; pp=vv['pose']; ix=np.where(valid)[0]
        if len(ix):
            first=ix[0]; pred0=np.repeat(pp[first:first+1],len(pp),axis=0); terr=np.linalg.norm(pred0[:,:3]-pp[:,:3],axis=1); oerr=np.full(len(pp),np.nan); oerr[valid]=np.degrees(quat_angle_rad(pred0[valid,3:],pp[valid,3:]))
            bucket_rows.append({'episode_index':rr['episode_index'],'object':nn,'visibility_bucket':'visible','frames':int(valid.sum()),'mean_translation_error_m':float(terr[valid].mean()),'mean_orientation_error_deg':float(oerr[valid].mean())})
        bucket_rows.append({'episode_index':rr['episode_index'],'object':nn,'visibility_bucket':'occluded','frames':int((~valid).sum()),'mean_translation_error_m':np.nan,'mean_orientation_error_deg':np.nan})
bucket_df=pd.DataFrame(bucket_rows); bucket_df.to_csv(paths['metrics']/'m4_visibility_buckets.csv',index=False)

cad={'status':'mesh/URDF identity available; image-to-world ICP blocked by missing extrinsic','objects':list(objs),'mean_translation_error_m':float(m4df.translation_error_m.mean()),'median_translation_error_m':float(m4df.translation_error_m.median()),'mean_orientation_error_deg':float(m4df.orientation_error_deg.mean()),'visibility_bucket_rows':bucket_df.to_dict(orient='records')}; write_json(paths['logs']/'m4_cad_tracking.json',cad); display(m4df.groupby('object')[['translation_error_m','orientation_error_deg']].agg(['mean','median','max']).round(4)); display(bucket_df.groupby('visibility_bucket')[['frames','mean_translation_error_m','mean_orientation_error_deg']].agg({'frames':'sum','mean_translation_error_m':'mean','mean_orientation_error_deg':'mean'}))

translation_error_m          ... orientation_error_deg         
                         mean  median  ...                median      max
object                                 ...                               
beige_cup              0.1517  0.1789  ...               22.4843  26.3048
blue_cup               0.0018  0.0021  ...                1.9628   4.6815

[2 rows x 6 columns]

,frames,mean_translation_error_m,mean_orientation_error_deg
visibility_bucket,,,
occluded,0,NaN,NaN
visible,8492,0.086095,38.140227


### Method comparison and physical plausibility

**What it does.** Combines measured runtime, memory, output sizes, Track 3 pose error, and raw/smoothed acceleration into one table.

**Input.** M1-M4 logs and mocap trajectory.

**Output.** `outputs/report/method_comparison.csv`, comparison figure, and acceleration plot.

**References.** V2D physical plausibility axis: https://nvidia-isaac.github.io/video_to_data/v2d_challenge/

In [13]:
from pathlib import Path
import json, sys, time, platform, subprocess, importlib.util
import matplotlib; matplotlib.use('Agg', force=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from IPython.display import display, Image
ROOT=Path.cwd()
if not (ROOT/'v2d_cpu_utils.py').exists(): ROOT=Path('/Users/luran/Desktop/V2D_Challenge')
sys.path.insert(0,str(ROOT))
from v2d_cpu_utils import *
paths=ensure_outputs()
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'font.size':9})

d=read_parquet(b['parquet']); objs=decode_track3_objects(d); dt=1/i3['fps']; ar=[]
for name,v in objs.items():
 valid=v['visible']; p=v['pose'][:,:3].copy(); p[~valid]=np.nan
 for j in range(3):
  g=np.where(np.isfinite(p[:,j]))[0]
  if len(g)>=2: p[:,j]=np.interp(np.arange(len(p)),g,p[g,j])
 raw=np.linalg.norm(finite_difference(p,dt,2),axis=1); sm=np.linalg.norm(finite_difference(smooth_signal(p),dt,2),axis=1)
 for i in np.where(valid)[0]: ar.append({'object':name,'frame':int(i),'acceleration_raw_m_s2':float(raw[i]),'acceleration_smoothed_m_s2':float(sm[i])})
adf=pd.DataFrame(ar); adf.to_csv(paths['metrics']/'track3_acceleration.csv',index=False); fig,ax=plt.subplots(figsize=(8,3.5)); [ax.plot(g.frame,g.acceleration_raw_m_s2,alpha=.3,label=f'{n} raw') or ax.plot(g.frame,g.acceleration_smoothed_m_s2,label=f'{n} smooth') for n,g in adf.groupby('object')]; ax.set_xlabel('frame index'); ax.set_ylabel('linear acceleration (m/s^2)'); ax.legend(ncol=2,fontsize=7); fig.tight_layout(); fig.savefig(paths['figs']/'track3_acceleration.png',dpi=140); plt.close(fig)
m1j=json.loads((paths['logs']/'m1_sfm_attempt.json').read_text()); rows=[{'method':'M1 classical SfM','track':'Track 1 + Track 3','wall_time_s':m1j['wall_time_s'],'peak_rss_mb':m1j['peak_rss_mb'],'geometry_metric':f"median matches T1={s1['median_matches']:.1f}, T3={s3['median_matches']:.1f}",'pose_metric':'N/A; metric SfM not completed','physical_metric':'2D feature stability'},{'method':'M2 depth fusion proxy','track':'Track 1 + Track 3','wall_time_s':m2['wall_time_s'],'peak_rss_mb':m2['peak_rss_mb'],'geometry_metric':f'points T1={len(p1)}, T3={len(p3)}','pose_metric':'N/A; pseudo-depth','physical_metric':'mask coverage'},{'method':'M3 TripoSR/InstantMesh fallback','track':'Track 1 + Track 3','wall_time_s':m3['wall_time_s'],'peak_rss_mb':m3['peak_rss_mb'],'geometry_metric':'ellipsoid extents','pose_metric':'N/A','physical_metric':'proxy only'},{'method':'M4 known CAD constant pose','track':'Track 3','wall_time_s':0.0,'peak_rss_mb':peak_rss_mb(),'geometry_metric':'released mesh/URDF identity','pose_metric':f"mean translation {cad['mean_translation_error_m']:.4f} m",'physical_metric':f"smoothed acceleration median {adf.acceleration_smoothed_m_s2.median():.4f} m/s^2"}]; comp=pd.DataFrame(rows); comp.to_csv(paths['report']/'method_comparison.csv',index=False); display(comp); fig,ax=plt.subplots(figsize=(8,3.5)); ax.bar(comp.method,comp.wall_time_s,color=['#4575b4','#74add1','#abd9e9','#d73027']); ax.set_ylabel('wall time (s)'); ax.tick_params(axis='x',rotation=20); fig.tight_layout(); fig.savefig(paths['figs']/'method_comparison.png',dpi=140); plt.close(fig)

,method,track,wall_time_s,peak_rss_mb,geometry_metric,pose_metric,physical_metric
0,M1 classical SfM,Track 1 + Track 3,0.447814,384.296875,"median matches T1=711.0, T3=572.0",N/A; metric SfM not completed,2D feature stability
1,M2 depth fusion proxy,Track 1 + Track 3,0.208852,440.296875,"points T1=8975, T3=14400",N/A; pseudo-depth,mask coverage
2,M3 TripoSR/InstantMesh fallback,Track 1 + Track 3,0.006184,440.437500,ellipsoid extents,N/A,proxy only
3,M4 known CAD constant pose,Track 3,0.000000,435.296875,released mesh/URDF identity,mean translation 0.0768 m,smoothed acceleration median 0.0204 m/s^2


### Execution boundary and qualitative conclusion

**What it does.** Prints exact model failures and CPU alternatives so measured results are separate from unavailable GPU methods.

**Input.** M1-M4 logs.

**Output.** `outputs/logs/method_boundary.json` and a reportable table.

**References.** HaMeR: https://arxiv.org/abs/2312.05251 ; FoundationPose: https://arxiv.org/abs/2312.08344.

In [15]:
from pathlib import Path
import json, sys, time, platform, subprocess, importlib.util
import matplotlib; matplotlib.use('Agg', force=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from IPython.display import display, Image
ROOT=Path.cwd()
if not (ROOT/'v2d_cpu_utils.py').exists(): ROOT=Path('/Users/luran/Desktop/V2D_Challenge')
sys.path.insert(0,str(ROOT))
from v2d_cpu_utils import *
paths=ensure_outputs()
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'font.size':9})

boundary=[{'requested_family':'M1 COLMAP SfM/MVS + Poisson','executed':'ORB adjacent feature tracks','reason':'camera intrinsics/masks/extrinsics are incomplete for metric reconstruction','status':'partial'},{'requested_family':'M2 Depth Anything V2 small / MoGe','executed':'foreground mask + pseudo-depth','reason':'weights are not local and network is disabled','status':'CPU fallback'},{'requested_family':'M3 TripoSR / InstantMesh','executed':'ellipsoid proxy','reason':'model package/weights unavailable under 16 GB budget','status':'CPU fallback'},{'requested_family':'M4 known-CAD ICP/template','executed':'released mesh/URDF + constant first-visible pose','reason':'Track 3 public release has no ego-to-mocap extrinsic','status':'pose baseline'},{'requested_family':'HaMeR / Fast-HaMeR hands','executed':'not run','reason':'no local hand-regression weights; Track 1 has no hand GT','status':'explicitly skipped'}]; display(pd.DataFrame(boundary)); write_json(paths['logs']/'method_boundary.json',boundary); print('This is a CPU feasibility study, not an official Track 1 Chamfer or Track 3 policy score.')

This is a CPU feasibility study, not an official Track 1 Chamfer or Track 3 policy score.


,requested_family,executed,reason,status
0,M1 COLMAP SfM/MVS + Poisson,ORB adjacent feature tracks,camera intrinsics/masks/extrinsics are incompl...,partial
1,M2 Depth Anything V2 small / MoGe,foreground mask + pseudo-depth,weights are not local and network is disabled,CPU fallback
2,M3 TripoSR / InstantMesh,ellipsoid proxy,model package/weights unavailable under 16 GB ...,CPU fallback
3,M4 known-CAD ICP/template,released mesh/URDF + constant first-visible pose,Track 3 public release has no ego-to-mocap ext...,pose baseline
4,HaMeR / Fast-HaMeR hands,not run,no local hand-regression weights; Track 1 has ...,explicitly skipped
